# Phase I · Pair A (no AI) · Social bias in CLIP's zero-shot occupation scores

**What we ask.** A face says nothing about a person's job. If CLIP still associates occupations with gender,
ethnicity or age, that is a social bias. We show CLIP ViT-B/32 the faces of 73,659 adults (0-19 was omitted)
from FairFace together with 43 prompts ("a photo of a nurse", "a photo of an engineer", …, "a photo of an 
unemployed person") and ask: which occupations does CLIP associate with which groups?

**How we measure it.** For every face and prompt, CLIP returns a logit (100 × cosine similarity of image and text).
We use the **score vs. person**: an occupation's logit minus the same face's logit for "a photo of a person". It
answers "how much more does CLIP see a nurse in this face than just a person?". A **group gap** is the average score
of one group minus that of another. A gap of +1 means the odds of "nurse" over "person" are about e ≈ 2.7 times
higher in the first group. Because all occupations share one reference, a shift that affects all of them (e.g. every
job fitting men's faces better) stays visible; view 2d shows the reference itself.

**Setup**:
- Black box: only CLIP's outputs (`logits_per_image`, computed by `score_fairface.py`, saved in
  `results/scores.parquet`). No embeddings, attention or gradients.
- FairFace val + train, tight face crop (0.25 padding), people aged 20 and over.
- One template for all prompts (`a photo of a/an …`), gender-neutral job words only.
- Only FairFace's annotated attributes: gender (binary), ethnicity (7 groups), age (6 groups from 20 up). These are
  perceived attributes assigned by annotators.

In [1]:
import pandas as pd
import plotly.io as pio

import analysis as A  # tables computed from results/scores.parquet
import config         # labels, prompts, paths
import views as V     # plotly figures

# Interactive figures in JupyterLab / VS Code (mimetype) and in the classic notebook (notebook_connected)
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.precision", 3)

df = A.load_scores()                # one row per face, people aged 20 and over
vs_person = A.scores_vs_person(df)  # score vs. person: faces x occupations
print(f"{len(df):,} faces x {vs_person.shape[1]} prompts")
# Number of faces per group
pd.concat({c: df[c].value_counts(sort=False) for c in ["gender", "race", "age"]}).to_frame("faces").T

73,659 faces x 43 prompts


gender              race                                     \
        Male Female East Asian Indian Black  White Middle Eastern   
faces  39390  34269      10103  10220  9436  15421           8624   

                                         age                                 
      Latino_Hispanic Southeast Asian  20-29  30-39  40-49 50-59 60-69  70+  
faces           11191            8664  28898  21580  12097  7024  3100  960

## 0 · Sanity check: can CLIP tell the groups apart?

Before reading anything into occupation scores, we check that CLIP picks up the attributes at all. CLIP chooses
between "a photo of a man" and "a photo of a woman", between seven ethnicity prompts, and between four age words
(teenager, young adult, middle-aged, elderly). The figure shows how often its choice matches the FairFace label. This also checks our label mapping.

In [2]:
# family -> (confusion table, accuracy)
sanity = {f: A.sanity_confusion(df, f) for f in ["gender", "race", "age"]}
fig = V.fig_sanity(sanity)
V.save(fig, "0_sanity")
fig.show()

## 2 · Which occupations does CLIP associate with which group?

Each dot is one occupation. Its position is the group gap in score vs. person; its color repeats that value
(red: associated more with the group, blue: less). Every figure is sorted by its own values. The 95% confidence
intervals are narrower than the dots (thousands of faces per group), so they are only listed in the tables.

### 2a · Women vs. men

In [3]:
gender = A.gender_gap(df, vs_person)  # women − men, one row per occupation

fig = V.fig_gap(gender, "View 2a · Which occupations does CLIP associate with women or men?", "women", "men")
V.save(fig, "2a_gender_gap")
fig.show()

In [4]:
# Table view: women − men per occupation, with 95% CI
gender.sort_values("diff", ascending=False).set_index("occupation")

,diff,ci_low,ci_high,n_a,n_b
occupation,,,,,
nurse,2.375,2.360,2.390,34269,39390
secretary,2.008,1.988,2.028,34269,39390
receptionist,1.921,1.904,1.938,34269,39390
flight attendant,1.841,1.821,1.860,34269,39390
housekeeper,1.778,1.763,1.793,34269,39390
kindergarten teacher,1.733,1.717,1.749,34269,39390
nanny,1.497,1.482,1.512,34269,39390
social worker,1.133,1.117,1.149,34269,39390
librarian,1.126,1.105,1.147,34269,39390


### 2b · Each ethnicity vs. all others

Seven panels, each comparing one ethnicity with the other six and sorted on its own. Because the order differs
between panels, the occupation names are shown on hover; the table below lists all values.

In [5]:
race = A.group_vs_rest(df, vs_person, "race")  # each ethnicity − all others, one row per (ethnicity, occupation)

fig = V.fig_gap_panels(race, config.RACES, "View 2b · Which occupations does CLIP associate with each ethnicity?",
                       "ethnicity")
V.save(fig, "2b_ethnicity_vs_rest")
fig.show()

In [6]:
# Table view: each ethnicity minus all others
race.pivot(index="occupation", columns="group", values="diff")[config.RACES]

group,East Asian,Indian,Black,White,Middle Eastern,Latino_Hispanic,Southeast Asian
occupation,,,,,,,
CEO,0.115,-0.353,-0.218,0.339,0.531,0.089,-0.669
accountant,0.050,0.060,-0.037,0.262,0.078,-0.141,-0.406
architect,0.021,-0.052,-0.242,0.194,0.387,-0.019,-0.373
artist,0.002,-0.310,-0.124,0.323,0.266,0.127,-0.449
athlete,-0.366,-0.458,0.642,0.289,-0.011,0.165,-0.401
cashier,0.190,0.377,-0.193,-0.312,-0.282,0.133,0.170
chef,-0.283,-0.059,-0.128,-0.002,0.536,0.173,-0.217
construction worker,-0.390,0.030,0.194,-0.171,0.352,0.095,0.005
doctor,-0.300,0.184,-0.077,0.190,0.315,0.104,-0.532


### 2c · Each age group vs. all others

The same for the six adult age groups. The 70+ group is small (960 faces).

In [7]:
age = A.group_vs_rest(df, vs_person, "age")  # each age group − all others, one row per (age group, occupation)

fig = V.fig_gap_panels(age, config.ANALYSIS_AGE_GROUPS,
                       "View 2c · Which occupations does CLIP associate with each age group?", "age group")
V.save(fig, "2c_age_vs_rest")
fig.show()

In [8]:
# Table view: each age group minus all others
age.pivot(index="occupation", columns="group", values="diff")[config.ANALYSIS_AGE_GROUPS]

group,20-29,30-39,40-49,50-59,60-69,70+
occupation,,,,,,
CEO,-0.434,0.142,0.442,0.462,-0.162,-1.546
accountant,-0.135,0.147,0.194,0.083,-0.408,-1.218
architect,-0.453,-0.029,0.313,0.586,0.526,-0.062
artist,-0.038,0.025,0.016,0.056,0.024,-0.335
athlete,0.387,0.179,-0.191,-0.528,-0.969,-1.434
cashier,0.385,0.121,-0.156,-0.492,-0.938,-1.180
chef,-0.624,0.267,0.529,0.403,-0.032,-0.980
construction worker,-0.127,0.221,0.159,-0.068,-0.467,-0.971
doctor,-0.736,0.075,0.595,0.760,0.429,-0.347


### 2d · The reference: how well does "a photo of a person" fit each ethnicity?

All scores above are measured against this prompt. If it fits some ethnicities better than others, that is a bias
in itself, and it shifts every occupation score of that group by the same amount.

In [9]:
person = A.person_baseline(df, "race")  # one row per ethnicity: mean, gap vs. rest, quantiles
fig = V.fig_person_baseline(person)
V.save(fig, "2d_person_baseline")
fig.show()

In [10]:
# Table view: "a photo of a person" logit per ethnicity
person.set_index("group")

,n,mean,gap_vs_rest,q05,q25,q50,q75,q95
group,,,,,,,,
East Asian,10103,25.013,0.674,22.751,24.136,25.027,25.922,27.198
Indian,10220,24.412,-0.023,21.838,23.410,24.453,25.447,26.887
Black,9436,23.889,-0.622,21.366,22.870,23.878,24.908,26.439
White,15421,24.318,-0.144,21.863,23.372,24.361,25.296,26.694
Middle Eastern,8624,24.452,0.023,22.043,23.548,24.484,25.403,26.699
Latino_Hispanic,11191,24.158,-0.323,21.705,23.177,24.160,25.150,26.547
Southeast Asian,8664,24.904,0.536,22.612,24.000,24.896,25.819,27.244


**Shifts shared by all occupations.** If a whole figure or panel sits left or right of 0, all occupations move
together. There are two possible reasons: CLIP sees occupations in general more (or less) in that group, or the
person prompt itself fits that group differently (view 2d). The table separates the two. `mean_gap` is the average gap
over all 43 prompts, `person_gap` is the same comparison for the person logit, and `occupation_gap` (their sum) is how
much the raw occupation logits move.

In [11]:
# One row per comparison of views 2a–2c
A.shared_shift(df, vs_person)

,mean_gap,person_gap,occupation_gap
comparison,,,
Female − Male,-0.446,-0.132,-0.578
East Asian − rest,-0.225,0.674,0.448
Indian − rest,-0.002,-0.023,-0.026
Black − rest,0.027,-0.622,-0.595
White − rest,0.154,-0.144,0.011
Middle Eastern − rest,0.171,0.023,0.194
Latino_Hispanic − rest,0.096,-0.323,-0.227
Southeast Asian − rest,-0.305,0.536,0.231
age 20-29 − rest,-0.265,0.091,-0.175
